# Multi-Week Telemetry Evaluation with AI Comments

This notebook executes the complete telemetry analysis pipeline with AI-powered maintenance insights.

**Pipeline Steps:**
1. **Baseline Computation** - Calculate historical percentile thresholds (P1, P5, P95, P99)
2. **Signal Evaluation** - Score each signal against baseline percentiles
3. **Component Aggregation** - Aggregate signals to component level + AI recommendations
4. **Machine Aggregation** - Aggregate components to machine level + AI health summaries
5. **Golden Layer Output** - Write evaluated data with AI comments to parquet files

**AI Integration:**
- Component-level: Technical maintenance recommendations for anomalous components
- Machine-level: Executive health summaries for fleet management
- Language: Spanish (for maintenance teams)
- Model: OpenAI GPT-4o-mini (cost-effective, fast)

**Data Source:**
- Input: `data/telemetry/silver/{client}/` (weekly parquet files)
- Output: `data/telemetry/golden/{client}/` (machine_status.parquet, classified.parquet)

**Configuration:**
- Client: CDA
- Percentile thresholds: P1/P99 (stricter than previous P2/P98)
- Signal thresholds: Alert=0.5, Anormal=1.2
- Component thresholds: Normal<0.20, Anormal≥0.60

In [ ]:
# Import required modules
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import warnings

# Import telemetry pipeline modules
from src.telemetry import data_loader, data_cleaner, baseline, scoring, aggregation, output_writer
from src.services.ai_comment_service import get_ai_service
from src.utils.logger import logger

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("✓ All modules imported successfully")
print(f"  Execution date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Step 1: AI Service Configuration

Check OpenAI API configuration and AI service availability.

In [ ]:
# Initialize AI service
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get AI service instance
ai_service = get_ai_service()

print("=" * 80)
print("AI SERVICE CONFIGURATION")
print("=" * 80)

if ai_service.enabled:
    print("\n✓ OpenAI API configured successfully")
    print(f"  Model: {ai_service.model}")
    print(f"  API Key: {'*' * 20}{os.getenv('OPENAI_API_KEY', '')[-8:]}")
    print(f"\nAI comments will be generated for:")
    print(f"  - Components with Alerta or Anormal status")
    print(f"  - All machines (detailed for non-Normal, standard for Normal)")
else:
    print("\n⚠️  OpenAI API not configured - AI comments will be NULL")
    print("\nTo enable AI comments:")
    print("  1. Copy .env.example to .env")
    print("  2. Add your OpenAI API key: OPENAI_API_KEY=sk-...")
    print("  3. Restart notebook kernel")
    print("\nNOTE: Pipeline will continue without AI comments")

## Step 2: Pipeline Configuration

Define the evaluation parameters and discover available weeks in silver layer.

In [ ]:
# Pipeline configuration
CLIENT = 'cda'
BASELINE_LOOKBACK_DAYS = 7 * 16  # 112 days (16 weeks)

# Discover available weeks in silver layer
silver_path = Path.cwd().parent / 'data' / 'telemetry' / 'silver' / CLIENT
silver_files = sorted(silver_path.glob('*.parquet'))

print("=" * 80)
print("PIPELINE CONFIGURATION")
print("=" * 80)

print(f"\nClient: {CLIENT}")
print(f"Silver layer path: {silver_path}")
print(f"Available files: {len(silver_files)}")

# Parse week numbers from filenames
weeks_to_process = []
for file_path in silver_files:
    # Expected format: Telemetry_Wide_With_States.parquet or week_XX_YYYY.parquet
    # For now, we'll need to read each file to get the week info
    try:
        df_sample = pd.read_parquet(file_path, columns=['Fecha'])
        if len(df_sample) > 0:
            # Get week and year from first date
            first_date = pd.to_datetime(df_sample['Fecha'].iloc[0])
            week_num = first_date.isocalendar()[1]
            year_num = first_date.year
            
            # Check if we already have this week
            if (week_num, year_num) not in weeks_to_process:
                weeks_to_process.append((week_num, year_num))
    except Exception as e:
        logger.warning(f"Could not parse {file_path.name}: {e}")

# Sort by year and week
weeks_to_process = sorted(weeks_to_process, key=lambda x: (x[1], x[0]))

print(f"\nWeeks discovered: {len(weeks_to_process)}")
if weeks_to_process:
    print(f"Range: Week {weeks_to_process[0][0]:02d}-{weeks_to_process[0][1]} to Week {weeks_to_process[-1][0]:02d}-{weeks_to_process[-1][1]}")
    print(f"\nWeeks to process:")
    for week, year in weeks_to_process:
        print(f"  - Week {week:02d}, {year}")
else:
    print("\n⚠️  No weeks found in silver layer!")
    print(f"   Expected location: {silver_path}")

## Step 3: Baseline Computation

Compute historical percentile baselines (P1, P5, P95, P99) for anomaly detection.

**Baseline Strategy:**
- Training window: 112 days (16 weeks) of historical data
- Percentiles computed per (unit, signal, operational_state)
- Fallback to aggregate if insufficient state-specific data
- Single baseline used for all evaluation weeks

In [ ]:
print("\n" + "=" * 80)
print("BASELINE COMPUTATION")
print("=" * 80)

if not weeks_to_process:
    print("\n⚠️  Cannot compute baseline - no weeks to process")
else:
    # Use first week for baseline computation
    BASELINE_EVAL_WEEK, BASELINE_EVAL_YEAR = weeks_to_process[0]
    BASELINE_VERSION = datetime.now().strftime('%Y%m%d')
    
    print(f"\nBaseline configuration:")
    print(f"  Evaluation week: {BASELINE_EVAL_WEEK:02d}-{BASELINE_EVAL_YEAR}")
    print(f"  Lookback period: {BASELINE_LOOKBACK_DAYS} days")
    print(f"  Baseline version: {BASELINE_VERSION}")
    
    # Load historical training data
    print(f"\nLoading historical data...")
    baseline_training_df = data_loader.load_baseline_training_window(
        client=CLIENT,
        evaluation_week=BASELINE_EVAL_WEEK,
        evaluation_year=BASELINE_EVAL_YEAR,
        lookback_days=BASELINE_LOOKBACK_DAYS
    )
    
    print(f"✓ Loaded {len(baseline_training_df):,} historical rows")
    print(f"  Date range: {baseline_training_df['Fecha'].min()} to {baseline_training_df['Fecha'].max()}")
    print(f"  Units: {baseline_training_df['Unit'].nunique()}")
    
    # Get signal columns
    signal_cols = data_loader.get_signal_columns(baseline_training_df)
    print(f"✓ Identified {len(signal_cols)} signal columns")
    
    # Clean baseline data
    print(f"\nCleaning baseline data...")
    baseline_training_clean = data_cleaner.clean_telemetry_data(baseline_training_df, signal_cols)
    print(f"✓ Cleaned: {len(baseline_training_clean):,} rows")
    
    # Compute percentiles (P1, P5, P95, P99)
    print(f"\nComputing baseline percentiles...")
    baseline_df = baseline.compute_baseline_percentiles(
        training_df=baseline_training_clean,
        signal_cols=signal_cols,
        baseline_date=BASELINE_VERSION
    )
    
    print(f"✓ Computed {len(baseline_df):,} baseline combinations")
    print(f"  Units with baselines: {baseline_df['Unit'].nunique()}")
    print(f"  Signals with baselines: {baseline_df['Signal'].nunique()}")
    
    # Show state breakdown
    state_breakdown = baseline_df['EstadoMaquina'].value_counts()
    print(f"\n  Baseline breakdown by operational state:")
    for state, count in state_breakdown.items():
        pct = (count / len(baseline_df)) * 100
        print(f"    {state}: {count} ({pct:.1f}%)")
    
    # Save baseline
    baseline_path = baseline.save_baseline(baseline_df, CLIENT)
    print(f"\n✓ Baseline saved to: {baseline_path}")
    
    # Load component mapping
    component_mapping = data_loader.load_component_mapping(CLIENT)
    print(f"✓ Loaded mapping for {len(component_mapping)} components")
    
    # Display sample baseline
    print(f"\nSample baseline thresholds:")
    sample_baseline = baseline_df.head(5)[['Unit', 'Signal', 'EstadoMaquina', 'P1', 'P5', 'P95', 'P99']]
    print(sample_baseline.to_string(index=False))

## Step 4: Multi-Week Evaluation Pipeline

Execute full pipeline for each week: scoring → component aggregation → machine aggregation → AI comments.

**Process per week:**
1. Load evaluation week data from silver layer
2. Clean and validate telemetry data
3. Score signals against baseline percentiles
4. Aggregate to component level + generate AI recommendations
5. Aggregate to machine level + generate AI health summaries
6. Write golden layer outputs (machine_status.parquet, classified.parquet)

In [ ]:
print("\n" + "=" * 80)
print("MULTI-WEEK EVALUATION PIPELINE")
print("=" * 80)

processing_summary = []

for idx, (week_num, year_num) in enumerate(weeks_to_process, 1):
    print(f"\n{'─' * 80}")
    print(f"[{idx}/{len(weeks_to_process)}] Processing Week {week_num:02d}, {year_num}")
    print(f"{'─' * 80}")
    
    try:
        # Step 1: Load evaluation week
        current_df = data_loader.load_evaluation_week(
            client=CLIENT,
            week=week_num,
            year=year_num
        )
        print(f"  ✓ Loaded {len(current_df):,} rows")
        print(f"    Date range: {current_df['Fecha'].min()} to {current_df['Fecha'].max()}")
        print(f"    Units: {current_df['Unit'].nunique()}")
        
        # Step 2: Clean data
        current_df_clean = data_cleaner.clean_telemetry_data(current_df, signal_cols)
        print(f"  ✓ Cleaned: {len(current_df_clean):,} rows (removed {len(current_df) - len(current_df_clean):,})")
        
        # Step 3: Score signals against baseline
        signal_evaluation_df = scoring.evaluate_signals(
            current_df=current_df_clean,
            baseline_df=baseline_df,
            signal_cols=signal_cols,
            component_mapping=component_mapping
        )
        print(f"  ✓ Evaluated {len(signal_evaluation_df):,} signal-unit combinations")
        
        # Show signal status distribution
        if not signal_evaluation_df.empty:
            status_dist = signal_evaluation_df['signal_status'].value_counts()
            print(f"    Signal status: Normal={status_dist.get('Normal', 0)}, "
                  f"Alerta={status_dist.get('Alerta', 0)}, "
                  f"Anormal={status_dist.get('Anormal', 0)}")
        
        # Step 4: Aggregate to components (AI recommendations generated here)
        component_df = aggregation.aggregate_to_components(
            signal_evaluation_df=signal_evaluation_df,
            component_mapping=component_mapping,
            current_df=current_df_clean,
            evaluation_week=week_num,
            evaluation_year=year_num,
            baseline_version=BASELINE_VERSION
        )
        print(f"  ✓ Aggregated to {len(component_df):,} component-unit combinations")
        
        # Track AI recommendations for components
        if 'ai_recommendation' in component_df.columns:
            ai_component_count = component_df['ai_recommendation'].notna().sum()
            print(f"    AI component recommendations: {ai_component_count}")
        
        # Step 5: Get expected fleet
        machine_status_path = Path.cwd().parent / 'data' / 'telemetry' / 'golden' / CLIENT / 'machine_status.parquet'
        expected_units = aggregation.get_expected_fleet(
            baseline_df=baseline_df,
            previous_machine_status_path=machine_status_path
        )
        
        # Step 6: Aggregate to machines (AI health summaries generated here)
        machine_df = aggregation.aggregate_to_machines(
            component_df=component_df,
            evaluation_week=week_num,
            evaluation_year=year_num,
            baseline_version=BASELINE_VERSION,
            expected_units=expected_units,
            component_mapping=component_mapping
        )
        print(f"  ✓ Aggregated to {len(machine_df):,} machines")
        
        # Show machine status distribution
        if not machine_df.empty:
            status_dist = machine_df['overall_status'].value_counts()
            print(f"    Machine status: Normal={status_dist.get('Normal', 0)}, "
                  f"Alerta={status_dist.get('Alerta', 0)}, "
                  f"Anormal={status_dist.get('Anormal', 0)}, "
                  f"InsufficientData={status_dist.get('InsufficientData', 0)}")
        
        # Track AI health summaries for machines
        if 'ai_health_summary' in machine_df.columns:
            ai_machine_count = machine_df['ai_health_summary'].notna().sum()
            print(f"    AI health summaries: {ai_machine_count}")
        
        # Step 7: Write golden layer outputs
        output_writer.write_golden_outputs(
            machine_df=machine_df,
            component_df=component_df,
            client=CLIENT
        )
        print(f"  ✓ Golden layer outputs written")
        
        # Track summary
        processing_summary.append({
            'week': week_num,
            'year': year_num,
            'raw_rows': len(current_df),
            'clean_rows': len(current_df_clean),
            'signal_evaluations': len(signal_evaluation_df),
            'components': len(component_df),
            'machines': len(machine_df),
            'machines_normal': (machine_df['overall_status'] == 'Normal').sum(),
            'machines_alerta': (machine_df['overall_status'] == 'Alerta').sum(),
            'machines_anormal': (machine_df['overall_status'] == 'Anormal').sum(),
            'machines_insufficient': (machine_df['overall_status'] == 'InsufficientData').sum(),
            'ai_component_comments': component_df['ai_recommendation'].notna().sum() if 'ai_recommendation' in component_df.columns else 0,
            'ai_machine_summaries': machine_df['ai_health_summary'].notna().sum() if 'ai_health_summary' in machine_df.columns else 0,
            'status': 'SUCCESS'
        })
        
    except Exception as e:
        print(f"  ✗ ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        processing_summary.append({
            'week': week_num,
            'year': year_num,
            'status': f'ERROR: {str(e)}'
        })

print(f"\n{'=' * 80}")
print(f"✓ PIPELINE EXECUTION COMPLETE")
print(f"{'=' * 80}")
print(f"\nProcessed: {len(processing_summary)} weeks")
successful = sum(1 for s in processing_summary if s['status'] == 'SUCCESS')
failed = len(processing_summary) - successful
print(f"  Success: {successful}")
print(f"  Failed: {failed}")

In [ ]:
# Display processing summary table
summary_df = pd.DataFrame(processing_summary)
summary_df['week_label'] = summary_df.apply(lambda x: f"W{x['week']:02d}-{x['year']}", axis=1)

print("\n" + "=" * 80)
print("PROCESSING SUMMARY")
print("=" * 80)

if 'machines' in summary_df.columns:
    print(f"\n{summary_df[['week_label', 'machines', 'machines_normal', 'machines_alerta', 'machines_anormal', 'machines_insufficient', 'ai_component_comments', 'ai_machine_summaries', 'status']].to_string(index=False)}")
else:
    print(f"\n{summary_df.to_string(index=False)}")

## Step 5: Golden Layer Validation

Verify outputs and validate data quality.

In [ ]:
print("=" * 80)
print("GOLDEN LAYER VALIDATION")
print("=" * 80)

# Load machine_status
machine_status_path = Path.cwd().parent / 'data' / 'telemetry' / 'golden' / CLIENT / 'machine_status.parquet'
classified_path = Path.cwd().parent / 'data' / 'telemetry' / 'golden' / CLIENT / 'classified.parquet'

if machine_status_path.exists():
    machine_status_df = pd.read_parquet(machine_status_path)
    
    print(f"\n✓ machine_status.parquet loaded")
    print(f"  Total records: {len(machine_status_df):,}")
    print(f"  Unique units: {machine_status_df['unit_id'].nunique()}")
    print(f"  Unique weeks: {machine_status_df[['evaluation_week', 'evaluation_year']].drop_duplicates().shape[0]}")
    
    # Check for duplicates
    duplicates = machine_status_df.duplicated(subset=['unit_id', 'evaluation_week', 'evaluation_year'], keep=False)
    if duplicates.any():
        print(f"  ⚠️  WARNING: {duplicates.sum()} duplicate records found!")
    else:
        print(f"  ✓ No duplicates (deduplication working)")
    
    # Check AI columns
    if 'ai_health_summary' in machine_status_df.columns:
        ai_count = machine_status_df['ai_health_summary'].notna().sum()
        print(f"  ✓ AI health summaries: {ai_count}/{len(machine_status_df)} ({ai_count/len(machine_status_df)*100:.1f}%)")
    else:
        print(f"  ⚠️  ai_health_summary column not found")
        
else:
    print(f"\n⚠️  machine_status.parquet not found at {machine_status_path}")

if classified_path.exists():
    classified_df = pd.read_parquet(classified_path)
    
    print(f"\n✓ classified.parquet loaded")
    print(f"  Total records: {len(classified_df):,}")
    print(f"  Unique units: {classified_df['unit_id'].nunique()}")
    print(f"  Unique components: {classified_df['component'].nunique()}")
    print(f"  Unique weeks: {classified_df[['evaluation_week', 'evaluation_year']].drop_duplicates().shape[0]}")
    
    # Check for duplicates
    duplicates = classified_df.duplicated(subset=['unit_id', 'component', 'evaluation_week', 'evaluation_year'], keep=False)
    if duplicates.any():
        print(f"  ⚠️  WARNING: {duplicates.sum()} duplicate records found!")
    else:
        print(f"  ✓ No duplicates (deduplication working)")
    
    # Check AI columns
    if 'ai_recommendation' in classified_df.columns:
        ai_count = classified_df['ai_recommendation'].notna().sum()
        print(f"  ✓ AI recommendations: {ai_count}/{len(classified_df)} ({ai_count/len(classified_df)*100:.1f}%)")
    else:
        print(f"  ⚠️  ai_recommendation column not found")
else:
    print(f"\n⚠️  classified.parquet not found at {classified_path}")

## Step 6: AI Comment Analysis

Analyze the quality and distribution of AI-generated maintenance insights.

In [ ]:
print("=" * 80)
print("AI COMMENT ANALYSIS")
print("=" * 80)

if machine_status_path.exists() and 'ai_health_summary' in machine_status_df.columns:
    print("\n📊 Machine-Level AI Health Summaries:")
    print(f"  Total machines: {len(machine_status_df):,}")
    print(f"  With AI summaries: {machine_status_df['ai_health_summary'].notna().sum():,}")
    print(f"  Without AI summaries: {machine_status_df['ai_health_summary'].isna().sum():,}")
    
    # AI summaries by status
    print(f"\n  AI summaries by machine status:")
    for status in ['Normal', 'Alerta', 'Anormal', 'InsufficientData']:
        status_df = machine_status_df[machine_status_df['overall_status'] == status]
        if len(status_df) > 0:
            with_ai = status_df['ai_health_summary'].notna().sum()
            print(f"    {status}: {with_ai}/{len(status_df)} ({with_ai/len(status_df)*100:.1f}%)")
    
    # Sample AI summaries
    print(f"\n📝 Sample AI Health Summaries:")
    
    # Show samples for each status
    for status in ['Anormal', 'Alerta', 'Normal']:
        status_samples = machine_status_df[
            (machine_status_df['overall_status'] == status) &
            (machine_status_df['ai_health_summary'].notna())
        ].head(2)
        
        if len(status_samples) > 0:
            print(f"\n  {status} machines:")
            for idx, row in status_samples.iterrows():
                print(f"    Unit: {row['unit_id']} | Week {row['evaluation_week']:02d}-{row['evaluation_year']}")
                print(f"    Score: {row['machine_score']:.2f} | Components affected: {row['components_anormal'] + row['components_alerta']}")
                print(f"    Summary: {row['ai_health_summary']}")
                print()
else:
    print("\n⚠️  No AI health summaries available for analysis")

if classified_path.exists() and 'ai_recommendation' in classified_df.columns:
    print("\n📊 Component-Level AI Recommendations:")
    print(f"  Total components: {len(classified_df):,}")
    print(f"  With AI recommendations: {classified_df['ai_recommendation'].notna().sum():,}")
    print(f"  Without AI recommendations: {classified_df['ai_recommendation'].isna().sum():,}")
    
    # AI recommendations by status
    print(f"\n  AI recommendations by component status:")
    for status in ['Normal', 'Alerta', 'Anormal', 'InsufficientData']:
        status_df = classified_df[classified_df['component_status'] == status]
        if len(status_df) > 0:
            with_ai = status_df['ai_recommendation'].notna().sum()
            print(f"    {status}: {with_ai}/{len(status_df)} ({with_ai/len(status_df)*100:.1f}%)")
    
    # Sample AI recommendations
    print(f"\n📝 Sample AI Component Recommendations:")
    
    # Show samples for problematic components
    problematic = classified_df[
        (classified_df['component_status'].isin(['Alerta', 'Anormal'])) &
        (classified_df['ai_recommendation'].notna())
    ].head(5)
    
    if len(problematic) > 0:
        for idx, row in problematic.iterrows():
            print(f"\n  Unit: {row['unit_id']} | Component: {row['component']} | Status: {row['component_status']}")
            print(f"  Score: {row['component_score']:.2f} | Triggering signals: {row['triggering_signals']}")
            print(f"  Recommendation: {row['ai_recommendation']}")
            print()
    else:
        print("\n  No problematic components with AI recommendations found")
else:
    print("\n⚠️  No AI component recommendations available for analysis")

## Step 7: Visualization & Trends

Visualize machine and component health trends over time.

In [ ]:
if machine_status_path.exists():
    # Prepare data for visualization
    machine_status_df['week_label'] = machine_status_df.apply(
        lambda x: f"{x['evaluation_year']}-W{x['evaluation_week']:02d}", axis=1
    )
    
    # Sort by year and week
    machine_status_df['sort_key'] = machine_status_df.apply(
        lambda x: (x['evaluation_year'], x['evaluation_week']), axis=1
    )
    machine_status_df = machine_status_df.sort_values('sort_key')
    
    # Machine status distribution over time
    machine_status_counts = machine_status_df.groupby(['week_label', 'overall_status']).size().reset_index(name='count')
    
    status_colors = {
        'Normal': '#28a745',
        'Alerta': '#ffc107',
        'Anormal': '#dc3545',
        'InsufficientData': '#6c757d'
    }
    
    fig_machines = px.bar(
        machine_status_counts,
        x='week_label',
        y='count',
        color='overall_status',
        title='Machine Status Distribution Over Time',
        labels={
            'week_label': 'Week-Year',
            'count': 'Number of Units',
            'overall_status': 'Overall Status'
        },
        color_discrete_map=status_colors,
        category_orders={'overall_status': ['Normal', 'Alerta', 'Anormal', 'InsufficientData']},
        barmode='stack',
        height=500
    )
    
    fig_machines.update_layout(
        xaxis_tickangle=-45,
        font=dict(size=12),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    
    fig_machines.show()
    print("✓ Machine status trend chart created")
else:
    print("⚠️  Cannot create visualizations - machine_status.parquet not found")

In [ ]:
if classified_path.exists():
    # Prepare component data
    classified_df['week_label'] = classified_df.apply(
        lambda x: f"{x['evaluation_year']}-W{x['evaluation_week']:02d}", axis=1
    )
    
    classified_df['sort_key'] = classified_df.apply(
        lambda x: (x['evaluation_year'], x['evaluation_week']), axis=1
    )
    classified_df = classified_df.sort_values('sort_key')
    
    # Component status distribution over time
    component_status_counts = classified_df.groupby(['week_label', 'component_status']).size().reset_index(name='count')
    
    fig_components = px.bar(
        component_status_counts,
        x='week_label',
        y='count',
        color='component_status',
        title='Component Status Distribution Over Time',
        labels={
            'week_label': 'Week-Year',
            'count': 'Number of Components',
            'component_status': 'Component Status'
        },
        color_discrete_map=status_colors,
        category_orders={'component_status': ['Normal', 'Alerta', 'Anormal', 'InsufficientData']},
        barmode='stack',
        height=500
    )
    
    fig_components.update_layout(
        xaxis_tickangle=-45,
        font=dict(size=12),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    
    fig_components.show()
    print("✓ Component status trend chart created")

In [ ]:
if machine_status_path.exists() and 'ai_health_summary' in machine_status_df.columns:
    # AI comment generation trend
    ai_by_week = machine_status_df.groupby('week_label').agg({
        'ai_health_summary': lambda x: x.notna().sum()
    }).rename(columns={'ai_health_summary': 'ai_summaries'}).reset_index()
    
    fig_ai = px.bar(
        ai_by_week,
        x='week_label',
        y='ai_summaries',
        title='AI Health Summaries Generated Per Week',
        labels={
            'week_label': 'Week-Year',
            'ai_summaries': 'Number of AI Summaries'
        },
        color_discrete_sequence=['#007bff'],
        height=400
    )
    
    fig_ai.update_layout(
        xaxis_tickangle=-45,
        font=dict(size=12)
    )
    
    fig_ai.show()
    print("✓ AI generation trend chart created")

## Step 8: Execution Summary

**Pipeline Execution Complete** ✅

### What was accomplished:
1. ✅ Computed baseline percentiles (P1, P5, P95, P99) from historical data
2. ✅ Evaluated all weeks in silver layer against baseline
3. ✅ Generated component-level AI maintenance recommendations
4. ✅ Generated machine-level AI health summaries
5. ✅ Wrote golden layer outputs with AI comments

### Outputs generated:
- **machine_status.parquet**: Machine-level health assessments with AI summaries
- **classified.parquet**: Component-level evaluations with AI recommendations
- **baseline files**: Historical percentile thresholds

### Next steps:
1. **Review AI comments** to validate quality and relevance
2. **Integrate with dashboard** to display AI insights
3. **Monitor costs** via OpenAI usage dashboard
4. **Fine-tune thresholds** based on observed false positive/negative rates
5. **Schedule regular runs** (weekly batch processing)

### AI Comment Statistics:
- Model used: GPT-4o-mini
- Language: Spanish
- Component comments: Generated for Alerta/Anormal components
- Machine summaries: Generated for all machines
- Cost: ~$1-2 USD/week for typical fleet

---

**Note**: To run this notebook again:
1. Ensure `OPENAI_API_KEY` is set in `.env` file
2. New weeks in silver layer will be automatically discovered
3. Previous outputs will be deduplicated (append mode)
4. AI comments regenerate on each run (not cached)